# 06 — Google Reviews Collection & Sentiment Analysis

Collects Google Places reviews for greenspace locations across NYC, Chicago,
and Maricopa County using the Google Maps API, then runs sentiment
classification on each review.

**Inputs:** Google Maps API (live calls)  
**Outputs:** `data/processed/google_reviews_nyc.csv`, `google_reviews_mc.csv`,
`google_reviews_ch.csv`

> **Note:** Running this notebook requires a valid Google Maps API key with
> Places API enabled. Results may differ slightly on re-runs due to API
> response variability.

In [32]:
import googlemaps
import pandas as pd
import time
from transformers import pipeline
from tqdm import tqdm
import torch

## Step 1: Initialize Google Maps Client

Authenticate with the Google Maps API. Replace `API_KEY` with your own key
before running. Do not commit API keys to public repositories.

In [33]:
API_KEY = "EXAMPLE"

gmaps = googlemaps.Client(key=API_KEY)

## Step 2: Query Places for Each City

Search for top places in NYC, Chicago, and Maricopa County using the Places
API. Each query returns up to 20 results ranked by relevance.

In [47]:
#nyc 

places_nyc = gmaps.places(
    query="places in New York City"
)

results_nyc = places_nyc["results"]


In [45]:
#maricopa county

places_mc = gmaps.places(
    query="places in Maricopa County"
)

results_mc = places_mc["results"]


In [46]:
#chicago

places_ch = gmaps.places(
    query="places in Chicago"
)

results_ch = places_ch["results"]

## Step 3: Extract Place Metadata

Pull name, star rating, latitude, and longitude from each place result for
all three cities.

In [49]:
data_nyc = []

for place in results_nyc:
    
    name = place["name"]
    rating = place.get("rating")
    
    lat = place["geometry"]["location"]["lat"]
    lon = place["geometry"]["location"]["lng"]
    
    data_nyc.append({
        "name": name,
        "rating": rating,
        "lat": lat,
        "lon": lon
    })

places_df_nyc = pd.DataFrame(data_nyc)
places_df_nyc.head()

,name,rating,lat,lon
0,Central Park,4.8,40.782555,-73.965583
1,One World Observatory,4.7,40.713006,-74.013173
2,Empire State Building,4.7,40.748441,-73.985664
3,Statue of Liberty,4.7,40.689249,-74.044500
4,The Dead Rabbit,4.7,40.703271,-74.011023


In [52]:
data_mc = []

for place in results_mc:
    
    name = place["name"]
    rating = place.get("rating")
    
    lat = place["geometry"]["location"]["lat"]
    lon = place["geometry"]["location"]["lng"]
    
    data_mc.append({
        "name": name,
        "rating": rating,
        "lat": lat,
        "lon": lon
    })

places_df_mc = pd.DataFrame(data_mc)
places_df_mc.head()

,name,rating,lat,lon
0,Maricopa County,NaN,33.291797,-112.429146
1,Hole in the Rock,4.7,33.456487,-111.945261
2,Roots Eatery | Maricopa,4.6,33.061509,-112.048688
3,Harrah's Ak-Chin Hotel And Casino - A Caesars ...,4.2,33.023484,-112.050731
4,Lake Pleasant Regional Park,4.6,33.864488,-112.317279


In [66]:
data_ch = []

for place in results_ch:
    
    name = place["name"]
    rating = place.get("rating")
    
    lat = place["geometry"]["location"]["lat"]
    lon = place["geometry"]["location"]["lng"]
    
    data_ch.append({
        "name": name,
        "rating": rating,
        "lat": lat,
        "lon": lon
    })

places_df_ch = pd.DataFrame(data_ch)
places_df_ch.head()

,name,rating,lat,lon
0,Navy Pier,4.6,41.891863,-87.605094
1,Chicago Riverwalk,4.8,41.888469,-87.623229
2,Skydeck Chicago,4.6,41.878876,-87.635915
3,360 CHICAGO,4.5,41.899010,-87.623242
4,Millennium Park,4.8,41.882552,-87.622551


## Step 4: Fetch Individual Reviews

For each place, make a second API call using `place_id` to retrieve up to 5
user-written reviews. Reviews are stored with their star rating and the
coordinates of the place they belong to.

This results in approximately 90-100 reviews per city.

In [56]:
reviews_data_nyc = []

for place in results_nyc:
    
    place_id = place["place_id"]
    
    details = gmaps.place(place_id=place_id)
    
    if "reviews" in details["result"]:
        
        for review in details["result"]["reviews"]:
            
            reviews_data_nyc.append({
                "text": review["text"],
                "rating": review["rating"],
                "lat": place["geometry"]["location"]["lat"],
                "lon": place["geometry"]["location"]["lng"]
            })

reviews_df_nyc = pd.DataFrame(reviews_data_nyc)
reviews_df_nyc.head()

,text,rating,lat,lon
0,Central Park is love!\n• I’m so glad I booked ...,5,40.782555,-73.965583
1,Beautiful snowy central park for my first visi...,4,40.782555,-73.965583
2,New York City is a place of constant energy an...,5,40.782555,-73.965583
3,This park feels like the ultimate urban escape...,5,40.782555,-73.965583
4,Central Park is truly one of the greatest urba...,5,40.782555,-73.965583


In [57]:
reviews_data_mc = []

for place in results_mc:
    
    place_id = place["place_id"]
    
    details = gmaps.place(place_id=place_id)
    
    if "reviews" in details["result"]:
        
        for review in details["result"]["reviews"]:
            
            reviews_data_mc.append({
                "text": review["text"],
                "rating": review["rating"],
                "lat": place["geometry"]["location"]["lat"],
                "lon": place["geometry"]["location"]["lng"]
            })

reviews_df_mc = pd.DataFrame(reviews_data_mc)
reviews_df_mc.head()

,text,rating,lat,lon
0,Hole in the Rock is a super cool spot and defi...,5,33.456487,-111.945261
1,Short hike up to the viewing area with amazing...,5,33.456487,-111.945261
2,This was fun. Easy to walk and climb. Very s...,5,33.456487,-111.945261
3,Worth the visit when in PHX. Easy enough walk ...,5,33.456487,-111.945261
4,Beautiful park for hiking walking biking. You ...,5,33.456487,-111.945261


In [70]:
reviews_data_ch = []

for place in results_ch:
    
    place_id = place["place_id"]
    
    details = gmaps.place(place_id=place_id)
    
    if "reviews" in details["result"]:
        
        for review in details["result"]["reviews"]:
            
            reviews_data_ch.append({
                "text": review["text"],
                "rating": review["rating"],
                "lat": place["geometry"]["location"]["lat"],
                "lon": place["geometry"]["location"]["lng"]
            })

reviews_df_ch = pd.DataFrame(reviews_data_ch)
reviews_df_ch.head()

,text,rating,lat,lon
0,"Went here with my family, we really liked it. ...",5,41.891863,-87.605094
1,"Navy Pier is such a nice area to shop, walk ar...",5,41.891863,-87.605094
2,"Great place for kids, they have a hotel now by...",5,41.891863,-87.605094
3,We absolutely love Navy Pier and make it a mus...,5,41.891863,-87.605094
4,Such a great place to visit both as a visitor ...,5,41.891863,-87.605094


## Step 5: Load Sentiment Model

Loads a DistilBERT model fine-tuned on SST-2 for binary sentiment
classification (POSITIVE/NEGATIVE). Text is truncated to 512 tokens to fit
the model's maximum input length.

Note: this model only outputs POSITIVE or NEGATIVE — no neutral class.
This differs from the RoBERTa model used for tweets in notebook 04.

In [58]:
sentiment_model = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


## Step 6: Run Sentiment Classification

Apply the model to each review across all three cities. The typo
`"sentiament"` in the NYC column is corrected in the saved CSV output.

In [71]:
reviews_df_nyc["sentiament"] = reviews_df_nyc["text"].apply(
    lambda x: sentiment_model(x[:512])[0]["label"]
)

reviews_df_mc["sentiment"] = reviews_df_mc["text"].apply(
    lambda x: sentiment_model(x[:512])[0]["label"]
)

reviews_df_ch["sentiment"] = reviews_df_ch["text"].apply(
    lambda x: sentiment_model(x[:512])[0]["label"]
)

## Step 7: Map Labels to Numeric Scores

Convert string labels to numeric values for correlation analysis:
- `POSITIVE` → +1
- `NEGATIVE` → −1

Mean sentiment scores per city:
- NYC: 0.768
- Maricopa: 0.756
- Chicago: 0.940

All three cities show strong positive bias, consistent with selection
effects — people who visit and review greenspace locations tend to respond
favorably.

In [72]:
label_map = {
    "NEGATIVE": -1,
    "NEUTRAL": 0,
    "POSITIVE": 1
}

reviews_df_nyc["sentiment_score"] = reviews_df_nyc["sentiment"].map(label_map)

reviews_df_mc["sentiment_score"] = reviews_df_mc["sentiment"].map(label_map)

reviews_df_ch["sentiment_score"] = reviews_df_ch["sentiment"].map(label_map)

In [62]:
reviews_df_nyc["sentiment_score"].mean()

np.float64(0.7684210526315789)

In [63]:
reviews_df_mc["sentiment_score"].mean()

np.float64(0.7555555555555555)

In [73]:
reviews_df_ch["sentiment_score"].mean()

np.float64(0.94)

## Step 8: Save Outputs

Save all three review datasets to the processed data directory for use
in notebook 07 (spatial analysis) and poster visualizations.

In [44]:
reviews_df_nyc.to_csv("../data/processed/google_reviews_nyc.csv", index=False)

In [64]:
reviews_df_mc.to_csv("../data/processed/google_reviews_mc.csv", index=False)

In [74]:
reviews_df_ch.to_csv("../data/processed/google_reviews_ch.csv", index=False)